# Notebook 02 – Làm sạch dữ liệu & Tạo đặc trưng (Data Cleaning & Feature Engineering)

**Nội dung thực hiện:**
1. **Làm sạch dữ liệu (6 bước):**
   - Chuẩn hóa `holiday` (`NaN` → `'None'`) và lan truyền nhãn `is_holiday = 1` cho cả 24 giờ trong ngày lễ.
   - Khử trùng lặp `date_time` (`keep='first'`), đổi nhiệt độ Kelvin sang độ C và lọc ngoại lai vật lý (`temp <= 0K`, `rain_1h >= 500 mm`).
   - Reindex trên lưới thời gian liên tục 1 giờ/lần để đảm bảo tính đúng độ trễ.
2. **Tạo đặc trưng (Feature Engineering):** Tạo 3 nhóm đặc trưng (Lịch & chu kỳ `sin/cos`, Thời tiết One-Hot, Đặc trưng trễ `lag_1h`, `lag_24h`, `roll_3h`) và kiểm tra tương quan.
3. **Lưu đầu ra:** Xuất file sạch `data/processed/traffic_features.parquet` và `data/processed/feature_sets.json`.

In [1]:
import json
import os
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 220)

RAW_PATH = os.path.join("..", "data", "raw", "metro_interstate_traffic_volume.csv")
PROCESSED_DIR = os.path.join("..", "data", "processed")
os.makedirs(PROCESSED_DIR, exist_ok=True)

TEST_START = pd.Timestamp("2018-01-01")

df = pd.read_csv(RAW_PATH)
df["date_time"] = pd.to_datetime(df["date_time"])

preprocessing_log = [{
    "moc_xu_ly": "1. Dữ liệu gốc (raw)",
    "mo_ta": "File CSV từ UCI ID 492",
    "so_dong": len(df),
}]
print("Số dòng ban đầu:", len(df))

Số dòng ban đầu: 48204


In [2]:
df["holiday"] = df["holiday"].fillna("None").astype(str).str.strip()

holiday_rows = df[df["holiday"] != "None"]
holiday_days = set(holiday_rows["date_time"].dt.normalize())

print("Số dòng có nhãn lễ   :", len(holiday_rows))
print("Số NGÀY lễ khác nhau :", len(holiday_days))
print(
    holiday_rows[["date_time", "holiday"]]
    .drop_duplicates("date_time")
    .head(10)
    .to_string(index=False)
)

Số dòng có nhãn lễ   : 61
Số NGÀY lễ khác nhau : 53
 date_time              holiday
2012-10-08         Columbus Day
2012-11-12         Veterans Day
2012-11-22     Thanksgiving Day
2012-12-25        Christmas Day
2013-01-01        New Years Day
2013-02-18 Washingtons Birthday
2013-05-27         Memorial Day
2013-07-04     Independence Day
2013-08-22           State Fair
2013-09-02            Labor Day


In [3]:
so_trung_hoan_toan = df.duplicated().sum()
so_trung_date_time = df.duplicated(subset=["date_time"]).sum()

df = df.sort_values("date_time", kind="mergesort")
df = df.drop_duplicates(subset=["date_time"], keep="first").reset_index(
    drop=True
)

assert df["date_time"].is_unique
preprocessing_log.append({
    "moc_xu_ly": "2. Sau loại trùng date_time",
    "mo_ta": (
        f"Loại {so_trung_date_time} dòng trùng giờ (gồm {so_trung_hoan_toan}"
        " dòng trùng cả 9 cột)"
    ),
    "so_dong": len(df),
})
print("Số dòng còn lại sau khi khử trùng giờ:", len(df))

Số dòng còn lại sau khi khử trùng giờ: 40575


In [5]:
df["temp_c"] = df["temp"] - 273.15
cot_so = ["temp_c", "rain_1h", "snow_1h", "clouds_all", "traffic_volume"]
desc_truoc = df[cot_so].describe().T

mask_temp = df["temp_c"] <= -50
mask_rain = df["rain_1h"] >= 500
mask_outlier = mask_temp | mask_rain

print(
    f"Loại {mask_temp.sum()} dòng temp_c <= -50°C và {mask_rain.sum()} dòng"
    " rain_1h >= 500mm"
)
df = df[~mask_outlier].reset_index(drop=True)
desc_sau = df[cot_so].describe().T

preprocessing_log.append({
    "moc_xu_ly": "3. Sau lọc ngoại lai vật lý",
    "mo_ta": (
        f"Loại {int(mask_temp.sum())} dòng lỗi nhiệt độ và"
        f" {int(mask_rain.sum())} dòng lỗi mưa"
    ),
    "so_dong": len(df),
})

so_sanh = pd.concat(
    {
        "TRƯỚC": desc_truoc[["mean", "std", "min", "max"]],
        "SAU": desc_sau[["mean", "std", "min", "max"]],
    },
    axis=1,
)
display(so_sanh.round(2))

Loại 0 dòng temp_c <= -50°C và 0 dòng rain_1h >= 500mm


TRƯỚC                               SAU                         
                   mean      std    min      max     mean      std    min      max
temp_c             8.24    13.09 -29.76    36.92     8.24    13.09 -29.76    36.92
rain_1h            0.08     0.77   0.00    55.63     0.08     0.77   0.00    55.63
snow_1h            0.00     0.01   0.00     0.51     0.00     0.01   0.00     0.51
clouds_all        44.21    38.68   0.00   100.00    44.21    38.68   0.00   100.00
traffic_volume  3291.08  1984.64   0.00  7280.00  3291.08  1984.64   0.00  7280.00

In [6]:
full_range = pd.date_range(
    start=df["date_time"].min(), end=df["date_time"].max(), freq="h"
)
missing_hours = full_range.difference(df["date_time"])

df_grid = df.set_index("date_time").reindex(full_range)
df_grid.index.name = "date_time"

preprocessing_log.append({
    "moc_xu_ly": "4. Lưới giờ liên tục (reindex)",
    "mo_ta": (
        f"Trải lên lưới 1h; phát hiện {len(missing_hours)} giờ khuyết (NaN)"
    ),
    "so_dong": len(df_grid),
})

# Tính đặc trưng trễ trên lưới giờ liên tục
df_grid["lag_1h"] = df_grid["traffic_volume"].shift(1)
df_grid["lag_24h"] = df_grid["traffic_volume"].shift(24)
df_grid["roll_3h"] = df_grid["traffic_volume"].shift(1).rolling(3).mean()

# Chỉ giữ lại các giờ có quan sát thực tế và đủ 3 đặc trưng trễ
df_clean = df_grid[df_grid["traffic_volume"].notna()].reset_index()
df_clean = df_clean.dropna(subset=["lag_1h", "lag_24h", "roll_3h"]).reset_index(
    drop=True
)
df_clean["traffic_volume"] = df_clean["traffic_volume"].astype(int)
df_clean["clouds_all"] = df_clean["clouds_all"].astype(int)

preprocessing_log.append({
    "moc_xu_ly": "5. Tập sạch cuối cùng (đủ lag)",
    "mo_ta": "Giữ giờ có quan sát thực tế và đủ lag_1h, lag_24h, roll_3h",
    "so_dong": len(df_clean),
})
print("Số dòng sạch hoàn chỉnh:", len(df_clean))

Số dòng sạch hoàn chỉnh: 34833


In [7]:
# 1. Nhóm đặc trưng Lịch & Chu kỳ
df_clean["hour"] = df_clean["date_time"].dt.hour
df_clean["day_of_week"] = df_clean["date_time"].dt.dayofweek
df_clean["month"] = df_clean["date_time"].dt.month
df_clean["is_weekend"] = (df_clean["day_of_week"] >= 5).astype(int)
df_clean["is_holiday"] = (
    df_clean["date_time"].dt.normalize().isin(holiday_days).astype(int)
)

gio_cao_diem = df_clean["hour"].between(7, 9) | df_clean["hour"].between(16, 18)
ngay_lam_viec = (df_clean["is_weekend"] == 0) & (df_clean["is_holiday"] == 0)
df_clean["is_rush_hour"] = (gio_cao_diem & ngay_lam_viec).astype(int)

df_clean["hour_sin"] = np.sin(2 * np.pi * df_clean["hour"] / 24)
df_clean["hour_cos"] = np.cos(2 * np.pi * df_clean["hour"] / 24)

# 2. Nhóm đặc trưng Thời tiết (One-Hot Encoding)
weather_dummies = pd.get_dummies(
    df_clean["weather_main"], prefix="wm", dtype=int
)
df_clean = pd.concat([df_clean, weather_dummies], axis=1)
WEATHER_ONEHOT_COLS = sorted(weather_dummies.columns.tolist())

# 3. Định nghĩa Bộ đặc trưng A (23 cột) và Bộ đặc trưng B (26 cột)
CALENDAR_FEATURES = [
    "hour",
    "day_of_week",
    "month",
    "is_weekend",
    "is_rush_hour",
    "is_holiday",
    "hour_sin",
    "hour_cos",
]
WEATHER_FEATURES = [
    "temp_c",
    "rain_1h",
    "snow_1h",
    "clouds_all",
] + WEATHER_ONEHOT_COLS
LAG_FEATURES = ["lag_1h", "lag_24h", "roll_3h"]

FEATURES_A = CALENDAR_FEATURES + WEATHER_FEATURES
FEATURES_B = FEATURES_A + LAG_FEATURES

print(
    f"Bộ đặc trưng A ({len(FEATURES_A)} biến) | Bộ đặc trưng B"
    f" ({len(FEATURES_B)} biến)"
)
df_clean[["date_time", "traffic_volume"] + CALENDAR_FEATURES + LAG_FEATURES].head()

Bộ đặc trưng A (23 biến) | Bộ đặc trưng B (26 biến)


,date_time,traffic_volume,hour,day_of_week,month,is_weekend,is_rush_hour,is_holiday,hour_sin,hour_cos,lag_1h,lag_24h,roll_3h
0,2012-10-03 15:00:00,5692,15,2,10,0,0,0,-0.707107,-0.707107,5337.0,5584.0,5107.000000
1,2012-10-03 16:00:00,6137,16,2,10,0,1,0,-0.866025,-0.500000,5692.0,6015.0,5305.333333
2,2012-10-03 21:00:00,2637,21,2,10,0,0,0,-0.707107,0.707107,2898.0,2361.0,3704.000000
3,2012-10-03 22:00:00,1777,22,2,10,0,0,0,-0.500000,0.866025,2637.0,1529.0,3042.000000
4,2012-10-03 23:00:00,1015,23,2,10,0,0,0,-0.258819,0.965926,1777.0,963.0,2437.333333


In [8]:
# 1. Chọn cột và lưu file Parquet
df_out = df_clean[
    ["date_time", "traffic_volume"] + FEATURES_B + ["weather_main", "holiday"]
]
df_out.to_parquet(
    os.path.join(PROCESSED_DIR, "traffic_features.parquet"), index=False
)

# 2. Lưu cấu hình Bộ đặc trưng A & B ra file JSON
feature_sets = {
    "CALENDAR_FEATURES": CALENDAR_FEATURES,
    "WEATHER_FEATURES": WEATHER_FEATURES,
    "LAG_FEATURES": LAG_FEATURES,
    "FEATURES_A": FEATURES_A,
    "FEATURES_B": FEATURES_B,
    "TEST_START": "2018-01-01",
}
with open(
    os.path.join(PROCESSED_DIR, "feature_sets.json"), "w", encoding="utf-8"
) as f:
  json.dump(feature_sets, f, indent=2)

# 3. Ghi thêm số dòng Train / Test và in bảng tổng kết
n_train = int((df_out["date_time"] < TEST_START).sum())
preprocessing_log += [
    {
        "moc_xu_ly": "Tập Train (< 2018-01-01)",
        "mo_ta": "Huấn luyện (81.4%)",
        "so_dong": n_train,
    },
    {
        "moc_xu_ly": "Tập Test (>= 2018-01-01)",
        "mo_ta": "Kiểm tra (18.6%)",
        "so_dong": len(df_out) - n_train,
    },
]

print("Đã lưu thành công! Kích thước bảng sạch:", df_out.shape)
pd.DataFrame(preprocessing_log)

Đã lưu thành công! Kích thước bảng sạch: (34833, 30)


,moc_xu_ly,mo_ta,so_dong
0,1. Dữ liệu gốc (raw),File CSV từ UCI ID 492,48204
1,2. Sau loại trùng date_time,Loại 7629 dòng trùng giờ (gồm 17 dòng trùng cả...,40575
2,3. Sau lọc ngoại lai vật lý,Loại 10 dòng lỗi nhiệt độ và 1 dòng lỗi mưa,40564
3,3. Sau lọc ngoại lai vật lý,Loại 0 dòng lỗi nhiệt độ và 0 dòng lỗi mưa,40564
4,4. Lưới giờ liên tục (reindex),Trải lên lưới 1h; phát hiện 11987 giờ khuyết (...,52551
5,5. Tập sạch cuối cùng (đủ lag),"Giữ giờ có quan sát thực tế và đủ lag_1h, lag_...",34833
6,Tập Train (< 2018-01-01),Huấn luyện (81.4%),28353
7,Tập Test (>= 2018-01-01),Kiểm tra (18.6%),6480
